**Note**: This is a basic implementation of CNN-based NST, which may require an amount of time for inference.

## **1. Import Libraries**
---

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import vgg19, VGG19_Weights

import cv2
from PIL import Image

import os
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
# from tqdm.notebook import tqdm

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## **2. Prepare Data**
---

In [3]:
# Normalization used in pre-trained VGG19
mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])

# Unnormalize tensor
def unnormalize(tensor):
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    tensor.clamp_(0, 1)
    return tensor

In [4]:
def load_image(img_path, size=(256,256)):
    img_transforms = transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    img        = Image.open(img_path).convert('RGB')
    img_tensor = img_transforms(img)
    img_tensor = img_tensor.unsqueeze(0).to(device, torch.float)
    
    return img_tensor

In [5]:
def plot_image(tensor, title=None, save=False, save_path=None):
    img = tensor.detach().clone().cpu().squeeze(0)
    img = unnormalize(img)
    img = transforms.ToPILImage()(img)

    if save:
        save_path = save_path if save_path is not None else '../results/transfered_img.png'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        img.save(save_path)

    plt.figure()
    plt.imshow(img)
    if title:
        plt.title(title, pad=10)
    plt.axis('off')
    plt.show()

In [7]:
def preprocess_frame(frame, size=(256,256)):
    img_transforms = transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])
    
    numpy_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image  = Image.fromarray(numpy_rgb)
    tensor_rgb = img_transforms(pil_image)
    tensor_rgb = tensor_rgb.unsqueeze(0).to(device, torch.float)
    return tensor_rgb

def postprocess_frame(tensor):
    tensor_clone = tensor.detach().clone().cpu().squeeze(0)
    tensor_rgb   = unnormalize(tensor_clone)
    
    numpy_rgb = tensor_rgb.permute(1, 2, 0).numpy()
    numpy_rgb = (numpy_rgb * 255).astype(np.uint8)
    numpy_bgr = cv2.cvtColor(numpy_rgb, cv2.COLOR_RGB2BGR)
    return numpy_bgr

## **3. Loss Function & Model**
---

In [9]:
class TransferLoss(nn.Module):
    def __init__(self, content_weight=1, style_weight=1e6):
        super().__init__()

        self.content_weight = content_weight
        self.style_weight   = style_weight

        self.content_layer = 'conv2_1'
        self.style_layers  = ['conv1_1', 'conv2_1', 'conv3_1', 'conv4_1', 'conv5_1']

        self.loss_fn = nn.MSELoss()

    def _gram_matrix(self, tensor): # Case: batch_size=1
        B, C, H, W = tensor.size()
        tensor     = tensor.view(B * C, H * W)
        G          = torch.mm(tensor, tensor.t())
        return G.div(B * C * H * W)

    def forward(self, content_features, style_features, target_features):
        self.content_loss = self.loss_fn(content_features[self.content_layer],
                                         target_features[self.content_layer])
        
        self.style_loss = sum(
            self.loss_fn(self._gram_matrix(style_features[layer]),
                         self._gram_matrix(target_features[layer]))
            for layer in self.style_layers
        )

        self.total_loss = self.content_weight * self.content_loss + self.style_weight * self.style_loss
        return self.total_loss

    def get_loss(self):
        return self.content_loss, self.style_loss, self.total_loss

In [10]:
class VGG19_FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = vgg19(weights=VGG19_Weights.DEFAULT).features.eval()
        self.model.to(device)
        for param in self.model.parameters():
            param.requires_grad_(False)

        # self.layers = {'0': 'conv1_1', '5': 'conv2_1', '10': 'conv3_1', '19': 'conv4_1', '21': 'conv4_2', '28': 'conv5_1'}
        self.layers = {'0': 'conv1_1', '5': 'conv2_1', '10': 'conv3_1', '19': 'conv4_1', '28': 'conv5_1'}
        
    def forward(self, x):
        features = {}
        for idx, layer in self.model._modules.items():
            x = layer(x.to(device))
            if idx in self.layers:
                features[self.layers[idx]] = x
        return features

## **4. Traning**
---

In [11]:
def NST_frame(content_img, style_features, extractor, transfer_loss_fn, steps=100, lr=0.01):
    target_img = content_img.clone().requires_grad_(True).to(device)
    # optimizer  = torch.optim.LBFGS([target_img], lr=lr)
    optimizer = torch.optim.AdamW([target_img], lr=lr)

    content_features = extractor(content_img)

    # def closure():
    #     optimizer.zero_grad()
    #     target_features = extractor(target_img)
    #     transfer_loss   = transfer_loss_fn(content_features, style_features, target_features)
    #     transfer_loss.backward()
    #     return transfer_loss

    for i in range(steps):
        # optimizer.step(closure)
        optimizer.zero_grad()
        target_features = extractor(target_img)
        transfer_loss   = transfer_loss_fn(content_features, style_features, target_features)
        transfer_loss.backward()
        optimizer.step()

    return target_img

In [ ]:
def NST_video(video_path, style_img_path, output_path, extractor, transfer_loss_fn, target_size='same', target_fps=10, steps_per_frame=100, lr=0.01):
    cap = cv2.VideoCapture(video_path)
    
    # Set size
    if target_size == 'same':
        target_size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    elif isinstance(target_size, tuple):
        target_size = target_size
    else:
        print('Wrong size!')
        return

    # Downsample fps (optional)
    fps         = int(cap.get(cv2.CAP_PROP_FPS))
    frame_skip  = max(1, fps // target_fps)
    total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Writer for saving ouput video  
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec
    out    = cv2.VideoWriter(output_path, fourcc, target_fps, target_size)

    # Get features
    style_img      = load_image(style_img_path, size=(target_size[1], target_size[0]))
    style_features = extractor(style_img)

    # Start style transfer
    with tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) // frame_skip), desc='Processing Video', unit='frame') as pbar:
        frame_cnt = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_cnt % frame_skip == 0:
                # Process
                content_img = preprocess_frame(frame, size=(target_size[1], target_size[0]))
                
                target_img = NST_frame(
                    content_img, style_features,
                    extractor, transfer_loss_fn,
                    steps=steps_per_frame, lr=lr
                )
                
                output_frame = postprocess_frame(target_img)
                out.write(output_frame)
    
                # Update pbar
                pbar.update(1)

            frame_cnt += 1
        
    # Release
    cap.release()
    out.release()

    print('Done!')

In [14]:
# Extractor and Loss Funcntion
extractor        = VGG19_FeatureExtractor()
transfer_loss_fn = TransferLoss(content_weight=1, style_weight=1e4)
transfer_loss_fn.to(device)

# NST
NST_video(
    video_path='/kaggle/input/tmp-dataset/videos/content_5s.mp4',
    style_img_path='/kaggle/input/tmp-dataset/images/style1.png',
    output_path='/kaggle/working/output_content_5s_style1.mp4',
    extractor=extractor,
    transfer_loss_fn=transfer_loss_fn,
    target_size=(1280,720), # W, H
    target_fps=30,
    steps_per_frame=100,
    lr=0.02
)

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:03<00:00, 167MB/s]


Processing Video:   0%|          | 0/173 [00:00<?, ?frame/s]

Done!
